# Glitch Art Catalog — Text Analysis

The medical images only have folder labels. The glitch catalog has text: every work's `method` says *how the distortion happened*.

1. Catalog counts
2. Intentional vs. accidental
3. `method` text → TF-IDF → top words per group
4. KMeans → groups of glitch processes
5. Save

⚠️ About 50 works is a small dataset — treat this as exploration.

## Setup

In [ ]:
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/data_utils.py" -O data_utils.py
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/text_utils.py" -O text_utils.py

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

from data_utils import distance_score, balance_score, silhouette_score
from text_utils import get_top_words

---
# 1. Catalog Counts

In [ ]:
catalog_df = pd.read_csv("data/glitch/glitch_credits.csv").fillna("")
print(catalog_df.shape)

for col in ["source_site", "license", "medium"]:
  print(catalog_df[col].value_counts())
  print()

Distortion types (comma lists, like counting motifs):

In [ ]:
all_types = []
for v in catalog_df["distortion_types"]:
  for t in v.split(","):
    if t.strip() != "":
      all_types.append(t.strip())

pd.Series(all_types).value_counts()

---
# 2. Intentional vs. Accidental

A first guess: `notes` says "accidental" → accidental; Glitch Gallery collects accidental bugs → accidental; Rhizome works are made as art → intentional.
If you add your own `intent` column to the spreadsheet, it's used instead.

In [ ]:
if "intent" not in catalog_df.columns:
  intents = []
  for note, site in zip(catalog_df["notes"], catalog_df["source_site"]):
    if "accidental" in note.lower() or site == "Glitch Gallery":
      intents.append("accidental")
    else:
      intents.append("intentional")
  catalog_df["intent"] = intents

catalog_df["intent"].value_counts()

---
# 3. Method Text → TF-IDF

Text = `medium` + `method`. With so few works, a word only needs to appear in 2 of them (`min_df=2`).

In [ ]:
catalog_df["method_text"] = catalog_df["medium"] + " " + catalog_df["method"]

method_vec = TfidfVectorizer(stop_words="english", min_df=2)
method_vct = method_vec.fit_transform(catalog_df["method_text"])

method_vocab = method_vec.get_feature_names_out()
print(method_vct.shape)
print(method_vocab)

Top words for each group: join the group's texts into one, vectorize, `get_top_words()`.

In [ ]:
for col in ["intent", "source_site"]:
  for value in catalog_df[col].unique():
    group_text = " ".join(catalog_df[catalog_df[col] == value]["method_text"])
    print(col, "=", value)
    print("  ", get_top_words(method_vec.transform([group_text]), method_vocab, n_words=12))

<span style="color:hotpink;">

**Interpretation — words**

- Accidental works are described with `[ e.g. bug, file, codec ]` — the language of failure.
- Intentional works with `[ e.g. pixel, website, users ]` — the language of process.

</span>

---
# 4. Groups of Glitch Processes

In [ ]:
method_dense = method_vct.toarray()

ks = list(range(2, 9))
sil_scores = []
bal_scores = []

for k in ks:
  pred = KMeans(n_clusters=k, random_state=1010).fit_predict(method_vct)
  sil_scores.append(silhouette_score(method_dense, pred))
  bal_scores.append(balance_score(pred))

plt.plot(ks, sil_scores, marker="o")
plt.title("silhouette")
plt.show()
plt.plot(ks, bal_scores, marker="o")
plt.title("balance")
plt.show()

In [ ]:
METHOD_K = 5

method_km = KMeans(n_clusters=METHOD_K, random_state=1010)
catalog_df["method_group"] = method_km.fit_predict(method_vct)

group_words = get_top_words(method_km.cluster_centers_, method_vocab, n_words=8)

for c in range(METHOD_K):
  in_c = catalog_df[catalog_df["method_group"] == c]
  print("GROUP", c, "-", len(in_c), "works -", in_c["intent"].value_counts().to_dict())
  print("  words:", group_words[c])
  print("  works:", list(in_c["title"]))
  print()

<span style="color:hotpink;">

**Interpretation — process groups**

| group | my name for it | intent mix |
|---|---|---|
| 0 | `[ e.g. codecs & file corruption ]` | `[ ]` |
| 1 | `[ ]` | `[ ]` |
| 2 | `[ ]` | `[ ]` |
| 3 | `[ ]` | `[ ]` |
| 4 | `[ ]` | `[ ]` |

- Groups with **both** intentional and accidental works: `[ ]` — the most interesting for the project.
- Closest to how medical artifacts happen (signal, compression, resizing): `[ ]`.

</span>

---
# 5. Save

In [ ]:
catalog_df.to_csv("data/clean/glitch_catalog_clean.csv", index=False)
print(catalog_df.shape)